In [9]:
%load_ext autoreload
%autoreload all

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:
import sys
from pathlib import Path

# Add the parent directory to sys.path so we can import src
sys.path.insert(0, str(Path().resolve().parent))

import polars as pl
from src.pipeline_hero import config

# read data

In [11]:
df = pl.read_csv(config.Config().input_path)
concept_df = pl.read_csv(config.Config().concept_dict).with_columns(pl.concat_str([pl.col("vocabulary_id"), pl.col("concept_code")], separator="/").alias("code"))

# build demographic table

In [12]:
# extract prefix and build dataframe
df_bd = df.filter(pl.col("code")== config.Vocabs.birth_date).select(["patient_id", "start"]).rename({"start": "birth_date"})
df_race = df.filter(pl.col("code").is_in(config.Vocabs.static_race)).select(["patient_id", "code"]).rename({"code": "race"})
df_ethnicity = df.filter(pl.col("code").is_in(config.Vocabs.static_ethnicity)).select(["patient_id", "code"]).rename({"code": "ethnicity"})
df_gender = df.filter(pl.col("code").is_in(config.Vocabs.static_gender)).select(["patient_id", "code"]).rename({"code": "gender"})

df_demo = df_bd.join(df_race, on="patient_id", how="left").join(df_ethnicity, on="patient_id", how="left").join(df_gender, on="patient_id", how="left")


# assumpations:
1. only take start time
2. no numerical values, but still keep the lab tests

In [13]:
df_data =(df
 .filter(
    ~pl.col("code").is_in(config.Vocabs.prefix_codes))
.join(
    df_demo,
    on="patient_id")
.with_columns(
    pl.col("start").str.to_datetime(
        format="%Y-%m-%d %H:%M:%S",
        strict=False
    ).alias("start_dt"),
    pl.col("birth_date").str.to_datetime(
        format="%Y-%m-%d %H:%M:%S",
        strict=False
    ).alias("birth_dt"))
.with_columns(
    day_gap_bd = (pl.col("start_dt") - pl.col("birth_dt")).dt.total_days()
)
.select(config.Columns.columns_traj + config.Columns.columns_static))


# build data agg by patient

In [16]:
df_sequence = (
    df_data
    .group_by("patient_id")
    .agg([
        pl.col("code").sort_by("start"),
        pl.col("day_gap_bd").sort_by("start"),
        pl.col("visit_id").sort_by("start"),
        pl.col("birth_date").first(),
        pl.col("race").first(),
        pl.col("ethnicity").first(),
        pl.col("gender").first()
    ])
)

df_sequence.write_parquet(config.Config().data_by_patient)

# build vocab in data

In [17]:
df_vocab = df.select("code").unique().with_columns(
    code_type = pl.col("code").str.split("/").list.get(0),
    code_value = pl.col("code").str.split("/").list.get(1))

In [18]:
df_vocab.join(concept_df, on = "code").select(["code", "code_type", "code_value", "concept_id", "concept_name"]).write_parquet(config.Tokenizer_vocabs.data_vocab)